# Anime Character Classifier (Notebook-Only)

Run cells in order. The notebook downloads the dataset via `kagglehub` and builds train/validation/test splits in memory (variables), not folders.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "tensorflow",
    "numpy",
    "Pillow",
    "scikit-learn",
    "kagglehub",
    "opencv-python-headless",
])

import json
import random

import numpy as np
import tensorflow as tf
from PIL import Image

from constants import ARTIFACTS_DIR, EPOCHS, MODEL_DIR, SEED
from data_utils import (
    download_dataset_path,
    list_class_dirs,
    list_images,
    make_dataset,
    resolve_source_dir,
    split_dataset_records,
)
from inference_utils import detect_primary_face, preprocess_pil_for_model, top_k_indices
from model_utils import ann_structure_text, build_model

MODEL_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATASET_PATH = download_dataset_path()
SOURCE_DIR = resolve_source_dir(DATASET_PATH)

print("Path to dataset files:", SOURCE_DIR)
print("Setup complete.")

## Dataset Check
This cell reads the input folders and confirms class distribution.

In [ ]:
if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source directory not found: {SOURCE_DIR}")

class_dirs = list_class_dirs(SOURCE_DIR)
if not class_dirs:
    raise ValueError(f"No class folders found in {SOURCE_DIR}")

summary = {d.name: len(list_images(d)) for d in class_dirs}
print(f"Source directory: {SOURCE_DIR.resolve()}")
print(f"Number of classes: {len(summary)}")
print(f"Total images: {sum(summary.values())}")
print("Images per class:")
for class_name, count in summary.items():
    print(f"  - {class_name}: {count}")

## Train/Validation/Test Split
This cell splits image paths into `train_records`, `val_records`, and `test_records` variables (no file copying).

In [ ]:
class_names, train_records, val_records, test_records, split_stats = split_dataset_records(class_dirs)

print("Dataset split completed (in memory).")
for split_name, records in (("train", train_records), ("val", val_records), ("test", test_records)):
    print(f"{split_name}: {len(records)} images")

## Model Definition
This cell builds the classifier model and writes the model summary artifact.

In [ ]:
num_classes = len(class_dirs)
model = build_model(num_classes)
model.summary()

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
summary_lines = []
model.summary(print_fn=lambda x: summary_lines.append(x))
(ARTIFACTS_DIR / "model_summary.txt").write_text("\n".join(summary_lines), encoding="utf-8")
print(f"Model summary saved to: {(ARTIFACTS_DIR / 'model_summary.txt').resolve()}")

## ANN Structure
This cell saves a plain-text description of the ANN/CNN architecture.

In [ ]:
ann_structure = ann_structure_text()

(ARTIFACTS_DIR / "ann_structure.txt").write_text(ann_structure, encoding="utf-8")
print(ann_structure)
print(f"ANN structure saved to: {(ARTIFACTS_DIR / 'ann_structure.txt').resolve()}")

## Training
This cell trains the model and stores the best and final checkpoints.

In [ ]:
train_ds = make_dataset(train_records, shuffle=True)
val_ds = make_dataset(val_records, shuffle=False)

(MODEL_DIR / "class_names.json").write_text(
    json.dumps(class_names, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

model = build_model(num_classes=len(class_names))
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / "best_model.keras"),
        monitor="val_accuracy",
        save_best_only=True,
    ),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
model.save(MODEL_DIR / "final_model.keras")
print(f"Training complete. Final model saved to: {(MODEL_DIR / 'final_model.keras').resolve()}")
print(f"Best model saved to: {(MODEL_DIR / 'best_model.keras').resolve()}")
print("Final epoch metrics:", {k: float(v[-1]) for k, v in history.history.items()})

## Evaluation
This cell evaluates model performance using the test split.

In [ ]:
test_ds = make_dataset(test_records, shuffle=False)

best_model_path = MODEL_DIR / "best_model.keras"
model_path = best_model_path if best_model_path.exists() else (MODEL_DIR / "final_model.keras")
trained_model = tf.keras.models.load_model(model_path)

loss, accuracy = trained_model.evaluate(test_ds, verbose=1)
print(f"Loaded model: {model_path.resolve()}")
print(f"Classes ({len(class_names)}): {class_names}")
print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {accuracy:.4f}")

## Single Image Prediction
Upload an image (or provide a path) to get predicted character name.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    uploaded_path = list(uploaded.keys())[0]
except Exception:
    uploaded_path = input("Enter image path to test: ").strip()

original_img = Image.open(uploaded_path).convert("RGB")
face_img, face_bbox = detect_primary_face(original_img)
arr = preprocess_pil_for_model(face_img)

class_names = json.loads((MODEL_DIR / "class_names.json").read_text(encoding="utf-8"))
probs = trained_model.predict(arr, verbose=0)[0]
pred_idx = int(np.argmax(probs))

top_indices = top_k_indices(probs, k=3)

print(f"Image: {uploaded_path}")
if face_bbox is None:
    print("Face detection: no face found, used full image.")
else:
    print(f"Face detection: used crop bbox={face_bbox}")
print(f"Predicted character: {class_names[pred_idx]}")
print(f"Confidence: {float(probs[pred_idx]):.6f}")
print("Top predictions:")
for idx in top_indices:
    print(f"  - {class_names[int(idx)]}: {float(probs[int(idx)]):.6f}")

## Save Weights
This final cell saves model weights for reuse in future sessions.

In [ ]:
weights_path = ARTIFACTS_DIR / "model_weights.weights.h5"
weights_path.parent.mkdir(parents=True, exist_ok=True)
trained_model.save_weights(weights_path)
print(f"Weights saved to: {weights_path.resolve()}")